In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
from utils.db_connector import get_engine

engine = get_engine('smart_store')
print("Import uspešan!")

Konekcija kreirana: localhost/smart_store
Import uspešan!


In [4]:
query = """
    SELECT
        f.row_id,
        f.order_id,
        f.sales,
        f.profit,
        f.quantity,
        f.discount,
        f.unit_price,
        f.shipping_cost,
        f.is_returned,
        c.customer_name,
        p.product_name,
        p.product_category,
        p.product_sub_category,
        p.product_base_margin,
        g.region,
        g.state_or_province,
        g.city,
        g.postal_code,
        cal.full_date AS order_date,
        cal.year,
        cal.month_name,
        cal.month_number,
        cal.quarter,
        seg.segment_name,
        op.order_priority,
        sm.ship_mode,
        m.manager_name
    FROM fact_orders f
    JOIN dim_customers     c   ON c.customer_key      = f.customer_id
    JOIN dim_product       p   ON p.product_key       = f.product_id
    JOIN dim_geography     g   ON g.geography_id      = f.geography_id
    JOIN dim_calendar      cal ON cal.date_id         = f.order_date_id
    JOIN dim_segment       seg ON seg.segment_id      = f.segment_id
    JOIN dim_orderpriority op  ON op.orderpriority_id = f.orderpriority_id
    JOIN dim_shipmode      sm  ON sm.shipmode_id      = f.shipmode_id
    JOIN dim_manager       m   ON m.manager_id        = f.manager_id
"""

df = pd.read_sql(query, engine)
print(f"Ucitano {len(df)} redova i {len(df.columns)} kolona")
df.head()

Ucitano 1951 redova i 27 kolona


,row_id,order_id,sales,profit,quantity,discount,unit_price,shipping_cost,is_returned,customer_name,...,postal_code,order_date,year,month_name,month_number,quarter,segment_name,order_priority,ship_mode,manager_name
0,448,3042,351.56,15.42,88,0.10,4.26,1.20,0,Jenny Gold,...,90041,2015-05-20,2015,May,5,2,Consumer,Medium,Regular Air,William
1,494,3397,1233.32,-20.25,65,0.10,19.98,4.00,0,Caroline Johnston,...,02129,2015-06-22,2015,June,6,2,Consumer,Medium,Regular Air,Erin
2,495,3397,47.31,-3.38,17,0.09,2.88,1.49,0,Caroline Johnston,...,02129,2015-06-22,2015,June,6,2,Consumer,Medium,Regular Air,Erin
3,694,4839,232.50,-160.38,35,0.05,6.48,8.73,0,Andrew Gonzalez,...,28206,2015-05-09,2015,May,5,2,Consumer,Critical,Regular Air,Sam
4,819,5920,2039.07,-121.75,14,0.07,155.06,7.07,0,Troy Cassidy,...,90004,2015-05-19,2015,May,5,2,Consumer,High,Regular Air,William


In [5]:
# Provera null vrednosti
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

if len(null_cols) == 0:
    print("Nema null vrednosti u DataFrame-u")
else:
    print(f"Pronadjene null vrednosti:")
    for col, count in null_cols.items():
        print(f"  {col}: {count} null vrednosti")
    
    # Zameni medijanom
    numeric_cols = df.select_dtypes(include='number').columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"  {col}: null zamenjeni medijanom ({median_val:.2f})")

print(f"\nUkupno redova: {len(df)}")

Nema null vrednosti u DataFrame-u

Ukupno redova: 1951


In [6]:
# Korelacija između kolona
numeric_df = df.select_dtypes(include='number').drop(columns=['row_id', 'order_id'])

corr_matrix = numeric_df.corr()

# Izvuci sve parove bez duplikata i dijagonale
corr_pairs = (
    corr_matrix
    .unstack()
    .reset_index()
    .rename(columns={'level_0': 'Column_1', 'level_1': 'Column_2', 0: 'Correlation'})
)

# Ukloni dijagonalu (korelacija kolone sa samom sobom = 1)
corr_pairs = corr_pairs[corr_pairs['Column_1'] != corr_pairs['Column_2']]

# Ukloni duplikate (A-B i B-A su isti par)
corr_pairs['pair'] = corr_pairs.apply(
    lambda x: tuple(sorted([x['Column_1'], x['Column_2']])), axis=1
)
corr_pairs = corr_pairs.drop_duplicates(subset='pair').drop(columns='pair')
corr_pairs['Correlation'] = corr_pairs['Correlation'].round(4)
corr_pairs = corr_pairs.sort_values('Correlation', ascending=False)

# Top 3 i Bottom 3
top3    = corr_pairs.head(3)
bottom3 = corr_pairs.tail(3)
result  = pd.concat([top3, bottom3])

print("Top 3 najvise korelisani parovi:")
print(top3.to_string(index=False))
print("\nTop 3 najmanje korelisani parovi:")
print(bottom3.to_string(index=False))

# Export
result.to_csv('exports/correlation.csv', index=False)
print("\nExportovano: exports/correlation.csv")

Top 3 najvise korelisani parovi:
    Column_1   Column_2  Correlation
month_number    quarter       0.8783
       sales unit_price       0.4436
       sales     profit       0.3644

Top 3 najmanje korelisani parovi:
           Column_1     Column_2  Correlation
product_base_margin         year          NaN
               year month_number          NaN
               year      quarter          NaN

Exportovano: exports/correlation.csv


In [7]:
# Top 10 najvrednijih customers po Sales
top10_customers = (
    df.groupby(['customer_name'])
    .agg(Total_Sales=('sales', 'sum'))
    .round(2)
    .sort_values('Total_Sales', ascending=False)
    .head(10)
    .reset_index()
)

print("Top 10 customers po Sales:")
print(top10_customers.to_string(index=False))

# Export bez indexa
top10_customers.to_csv('exports/top_customers.csv', index=False)
print("\nExportovano: exports/top_customers.csv")

Top 10 customers po Sales:
    customer_name  Total_Sales
Kristine Connolly     50475.31
 Nina Horne Kelly     48295.12
     Toni Swanson     32194.12
 Rosemary O'Brien     29916.01
      Yvonne Mann     28779.13
           Lee Xu     20640.35
     Erin Ballard     20565.99
     Tammy Raynor     18642.71
       Neal Wolfe     17390.24
    Andrew Levine     16792.21

Exportovano: exports/top_customers.csv


In [8]:
# Pomocna funkcija za pivot tabelu
def create_pivot(df, group_col):
    pivot = (
        df.groupby(group_col)
        .agg(
            Avg_Discount=('discount', 'mean'),
            Avg_Shipping_Cost=('shipping_cost', 'mean'),
            Total_Profit=('profit', 'sum'),
            Total_Sales=('sales', 'sum')
        )
        .round(2)
        .reset_index()
    )
    return pivot

# Kreiraj sve pivot tabele
pivot_order_priority    = create_pivot(df, 'order_priority')
pivot_segment           = create_pivot(df, 'segment_name')
pivot_category          = create_pivot(df, 'product_category')
pivot_sub_category      = create_pivot(df, 'product_sub_category')
pivot_state             = create_pivot(df, 'state_or_province')

# Prikazi
print("Order Priority:")
print(pivot_order_priority.to_string(index=False))
print("\nCustomer Segment:")
print(pivot_segment.to_string(index=False))
print("\nProduct Category:")
print(pivot_category.to_string(index=False))

# Export
pivot_order_priority.to_csv('exports/pivot_order_priority.csv', index=False)
pivot_segment.to_csv('exports/pivot_segment.csv', index=False)
pivot_category.to_csv('exports/pivot_category.csv', index=False)
pivot_sub_category.to_csv('exports/pivot_sub_category.csv', index=False)
pivot_state.to_csv('exports/pivot_state.csv', index=False)

print("\nSvih 5 pivot tabela exportovano u exports/ folder!")

Order Priority:
order_priority  Avg_Discount  Avg_Shipping_Cost  Total_Profit  Total_Sales
      Critical          0.05              13.10      38293.48    441409.38
          High          0.05              13.32      36500.43    310095.48
           Low          0.05              13.83      35414.96    379127.34
        Medium          0.05              12.69      43370.17    370078.81
 Not Specified          0.05              11.90      70486.38    420026.22

Customer Segment:
  segment_name  Avg_Discount  Avg_Shipping_Cost  Total_Profit  Total_Sales
      Consumer          0.05              12.12      49882.70    398177.72
     Corporate          0.05              12.94      54444.28    657784.53
   Home Office          0.05              12.43      54433.48    464481.04
Small Business          0.05              14.58      65304.96    400293.94

Product Category:
product_category  Avg_Discount  Avg_Shipping_Cost  Total_Profit  Total_Sales
       Furniture          0.05              